**Helpers Methods**

In [ ]:
# Helper methods for NSE data processing
import pandas as pd
import requests, zipfile, io, datetime as dt, os
from typing import Tuple
import nse_workday
import re

DATA_STORE_DIR = r"C:\Users\bkaly\Documents\Trading\fo"

def get_holidays_list(date: dt.date) -> bool:
    # Get the current date and time
    current_year = dt.datetime.now().year
    # Get holidays in a specific date range
    start_date = f"01-01-{current_year}"  # Format: DD-MM-YYYY
    end_date = f"31-12-{current_year}"
    holidays = nse_workday.get_holidays_list(start_date=start_date, end_date=end_date)
    
    # Alternatively, check if a specific date is a holiday
    date_to_check = date  # e.g., Independence Day
    is_holiday = nse_workday.isHoliday(date_to_check)
    return bool(is_holiday)


def extract_symbol_and_date(contract_string):
    """
    Extracts the symbol and expiry date from a futures or option contract string.

    Args:
    contract_string (str): The input string like "FUTSTKASTRAL24-FEB-2026" or "OPTSTKASTRAL27-JAN-2026CE1640"

    Returns:
    tuple: (symbol, date) or (None, None) if no match
    """
    if not isinstance(contract_string, str):
        return None, None

    # FUT format e.g., FUTSTKASTRAL24-FEB-2026
    pattern_fut = r'FUTSTK([A-Z]+)(\d{2})-([A-Z]{3})-(\d{4})'
    m = re.match(pattern_fut, contract_string, re.I)
    if m:
        symbol = m.group(1).upper()
        day = m.group(2)
        month = m.group(3).upper()
        year = m.group(4)
        date = f"{day}-{month}-{year}"
        return symbol, date

    # OPT format e.g., OPTSTKASTRAL27-JAN-2026CE1640 or OPTSTKASTRAL27-JAN-2026PE850
    pattern_opt = r'OPTSTK([A-Z]+)(\d{2})-([A-Z]{3})-(\d{4})(CE|PE)?(\d+)?'
    m = re.match(pattern_opt, contract_string, re.I)
    if m:
        symbol = m.group(1).upper()
        day = m.group(2)
        month = m.group(3).upper()
        year = m.group(4)
        date = f"{day}-{month}-{year}"
        opType = m.group(5).upper()
        return symbol, date, opType

    return None, None

def check_file_exists(file_path):
    """
    Checks if a file exists at the given path.
    
    Args:
    file_path (str): The path to the file to check.
    
    Returns:
    bool: True if the file exists, False otherwise.
    """
    return os.path.exists(file_path) and os.path.isfile(file_path)

def download_fo_zip(trade_date: dt.date) -> pd.DataFrame:
    req_date = trade_date.strftime("%d-%b-%Y")
    file_date = trade_date.strftime("%d%m%y")
    url = f"https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A%22F%26O%20-%20Bhavcopy%20(fo.zip)%22%2C%22type%22%3A%22archives%22%2C%22category%22%3A%22derivatives%22%2C%22section%22%3A%22equity%22%7D%5D&date={req_date}&type=equity&mode=single"
     # Use headers to simulate a browser
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Referer': 'https://www.nseindia.com/'
    }
    response = requests.get(url, headers=headers, timeout=10)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        with z.open(f"fo{file_date}.csv") as f:
            df = pd.read_csv(f)
            df[['symbol', 'date']] = df['CONTRACT_D'].apply(lambda x: pd.Series(extract_symbol_and_date(x)))
        
        with z.open(f"op{file_date}.csv") as o:
            dfOption = pd.read_csv(o)
            dfOption[['symbol', 'date','optiontype']] = dfOption['CONTRACT_D'].apply(lambda x: pd.Series(extract_symbol_and_date(x)))
    dfOption.to_csv(os.path.join(DATA_STORE_DIR, f"op_{file_date}.csv"), index=False)
    df.to_csv(os.path.join(DATA_STORE_DIR, f"fo_{file_date}.csv"), index=False)
    return df

**Download & Generate FO**

In [ ]:
from datetime import timedelta
# Set the current date (as provided: December 09, 2025)
current_date = dt.date(2025, 12, 10)

# Calculate the start date (30 days ago)
start_date = current_date - timedelta(days=30)

# Iterate through the last 30 days (inclusive of today)
current = start_date
for i in range(31):  # 30 days back + today = 31 iterations
    file_date = current.strftime("%d%m%y")
    file_path = os.path.join(DATA_STORE_DIR, f"fo_{file_date}.csv")
    option_file_path = os.path.join(DATA_STORE_DIR, f"op_{file_date}.csv")
    # Check if the file exists and if the date is not a holiday
    if not check_file_exists(file_path) and not check_file_exists(option_file_path) and not get_holidays_list(current):
        download_fo_zip(current)
    current += timedelta(days=1)

**Implement AI to find the best stocks**

In [ ]:
#install semantic kernel package
%pip install semantic-kernel

In [ ]:
#install openai package
%pip install openai

In [ ]:
from openai import AsyncOpenAI
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

In [ ]:
## Create LLM Client
client = AsyncOpenAI(base_url="https://api.groq.com/openai/v1", api_key="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")

In [ ]:
# Initialize the kernel
kernel = Kernel()
kernel.add_service(OpenAIChatCompletion(ai_model_id="llama-3.1-8b-instant", async_client=client), overwrite=True)

In [ ]:
# call the OpenAIChatCompletion Service
from semantic_kernel.connectors.ai.chat_completion_client_base import ChatCompletionClientBase

chat_completion_service = kernel.get_service(type=ChatCompletionClientBase)

In [ ]:
from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
executionSettings = OpenAIChatPromptExecutionSettings(
    max_tokens=2000,
    temperature=0.01,
)

In [ ]:
# Create Chat Message History
from semantic_kernel.contents.chat_history import ChatHistory
mychat_history = ChatHistory()
mychat_history.add_system_message("""
                                  You are an expert intraday trading analyst specializing in Indian equity futures (NIFTY, BANKNIFTY, FINNIFTY, and individual stock futures on NSE). Your goal is to recommend exactly ONE best stock future for an intraday trade (buy or sell) based on real-time market conditions and the data provided. Always prioritize high-probability setups with strong risk-reward ratios""")

In [ ]:
import glob
import asyncio
from semantic_kernel.connectors.ai.chat_completion_client_base import ChatCompletionClientBase

def read_fo_data_from_folder(folder_path: str) -> pd.DataFrame:
    """
    Reads all FO CSV files from the specified folder and combines them into a single DataFrame.
    
    Args:
    folder_path (str): Path to the folder containing FO CSV files
    
    Returns:
    pd.DataFrame: Combined dataframe from all CSV files
    """
    all_files = glob.glob(os.path.join(folder_path, "fo_*.csv"))
    
    if not all_files:
        print(f"No FO files found in {folder_path}")
        return pd.DataFrame()
    
    dataframes = []
    for file in sorted(all_files):
        try:
            df = pd.read_csv(file)
            dataframes.append(df)
            print(f"Loaded: {os.path.basename(file)}")
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        return combined_df
    return pd.DataFrame()

async def analyze_fo_data_with_llm(data: pd.DataFrame) -> str:
    """
    Passes FO data to the LLM for analysis based on the system message.
    
    Args:
    data (pd.DataFrame): The FO data to analyze
    
    Returns:
    str: LLM response with trading recommendations
    """
    if data.empty:
        return "No data to analyze"
    
    # Prepare data summary for LLM
    data_summary = f"""
    Total records: {len(data)}
    
    Data columns: {', '.join(data.columns.tolist())}
    
    Sample data (first 5 records):
    {data.head().to_string()}
    
    Data statistics:
    {data.describe().to_string()}
    
    Unique symbols: {data['symbol'].unique().tolist() if 'symbol' in data.columns else 'N/A'}
    """
    
    # Add user message with data to chat history
    mychat_history.add_user_message(f"Please analyze the following FO data and recommend the best intraday trading opportunity:\n\n{data_summary}")
    
    # Get response from LLM
    response = await chat_completion_service.get_chat_message_content(
        chat_history=mychat_history,
        settings=executionSettings
    )
    
    # Add assistant response to chat history for context
    mychat_history.add_assistant_message(str(response))
    
    return str(response)

# Read data from folder
print("Reading FO data from folder...")
fo_data = read_fo_data_from_folder(DATA_STORE_DIR)

if not fo_data.empty:
    print(f"\nTotal records loaded: {len(fo_data)}")
    print("\nSending data to LLM for analysis...")
    
    # Run async analysis
    llm_response = await analyze_fo_data_with_llm(fo_data)
    print("\n=== LLM Analysis Results ===")
    print(llm_response)
else:
    print("No data to analyze")

In [ ]:
def download_and_load_fo(date: dt.date) -> pd.DataFrame:
    """Download NSE FO bhav-copy for a date and return DF or raise."""
 # Format the date
    date_str = date.strftime('%Y%m%d')  # YYYYMMDD
    filename = f'BhavCopy_NSE_FO_0_0_0_{date_str}_F_0000.csv.zip'
    url = f'https://nsearchives.nseindia.com/content/fo/{filename}'

    print(f"Fetching {url}")
    
    # Use headers to simulate a browser
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Referer': 'https://www.nseindia.com/'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            z = zipfile.ZipFile(io.BytesIO(response.content))
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f)
            return df
        else:
            print(f"Failed to download. HTTP Status Code: {response.status_code}")
            print("Possible reasons: Non-trading day, wrong URL format, or file not available yet.")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error occurred: {e}")
        return pd.DataFrame()

In [ ]:
from datetime import timedelta

DATA_STORE_DIR = r"C:\Users\bkaly\Documents\Trading\bhavcopy"

# Set the current date (as provided: December 09, 2025)
current_date = dt.date(2025, 12, 10)

# Calculate the start date (30 days ago)
start_date = current_date - timedelta(days=30)

# Iterate through the last 30 days (inclusive of today)
current = start_date
for i in range(31):  # 30 days back + today = 31 iterations
    file_date = current.strftime("%d%m%y")
    file_path = os.path.join(DATA_STORE_DIR, f"{file_date}_bhavcopy.csv")
    if not check_file_exists(file_path) and not get_holidays_list(current):
        df = download_and_load_fo(current)
        df.to_csv(os.path.join(DATA_STORE_DIR, f"{file_date}_bhavcopy.csv"), index=False)
    current += timedelta(days=1)